# 3. Dados climaticos (BR-DWGD) -- Soja no Parana

Substitui a etapa de download/formatacao do ERA5 (`3. Climatic data.ipynb`, pipeline de
milho) pela extracao direta dos arquivos locais do **BR-DWGD** (ja baixados em
`D:\_py\Clima_AgroIA\data-raw\xavier-data`, um NetCDF por variavel: `Tmax`, `Tmin`,
`RH`, `u2`, `Rs`, `pr`, cobrindo 2001-2025).

Para cada municipio do PR:
1. extrai a serie diaria das 6 variaveis do ponto da grade BR-DWGD mais proximo do
   centroide do municipio (`util/brdwgd.py`);
2. converte para as variaveis/unidades que o WOFOST espera (`IRRAD, TMIN, TMAX, VAP, RAIN, WIND`);
3. mescla com a produtividade anual do IBGE (etapa 2), replicando o valor anual do
   `yield (kg/ha)` em todos os dias daquele ano-safra (mesma convencao usada nos arquivos
   `point_id_*.nc` do pipeline de milho);
4. salva um arquivo `point_id_<cod_municipio>.nc` em `inputs/data/soja_pr/completo/`.

Aqui usamos o **codigo IBGE do municipio como `point_id`** (em vez de um hash opaco), o
que deixa o dataset autoexplicativo -- e totalmente compativel com
`SensitivityAnalyzer.NetCDFDataLoader`, que so espera o padrao de nome `point_id_*.nc` e
le `cod_municipio` a partir do restante do nome do arquivo.

O `dyield` (produtividade destendenciada) ainda **nao** e preenchido aqui -- isso e feito
na etapa 5 (Detrending), que abre estes mesmos arquivos e adiciona a variavel `dyield` +
atributos `beta_0`/`beta_1`.


In [1]:
import os
import sys

import numpy as np
import pandas as pd
import xarray as xr

sys.path.append(os.path.join(os.getcwd(), 'util'))
from util.brdwgd import BRDWGDLoader
from util.utils_soja_pr import setup_paths_soja_pr, safra_ano_colheita

paths = setup_paths_soja_pr()

df_locs = pd.read_excel(paths['COORDINATES'], sheet_name='SIDRA-ids')
df_yield = pd.read_excel(paths['COORDINATES'], sheet_name='yield_PR_soja')
df_locs['cod_municipio'] = df_locs['cod_municipio'].astype(str)
df_yield['cod_municipio'] = df_yield['cod_municipio'].astype(str)

# Janela climatica: um pouco antes da primeira safra (para cobrir o ciclo completo da
# safra 2015/16, semeada em out/2015) ate um pouco depois da ultima safra (2024/25).
DATA_INICIO = "2015-09-01"
DATA_FIM = "2025-06-30"

print(f"{len(df_locs)} municipios do PR carregados.")


399 municipios do PR carregados.


In [2]:
paths

{'BASE': 'd:\\_py\\AgroIA_prod',
 'DATA': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr',
 'COORDINATES': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr\\coordinates_pr.xlsx',
 'COMPLETO': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soja_pr\\completo',
 'AGRO': 'd:\\_py\\AgroIA_prod\\inputs\\data\\agro\\agro_soybean_pr.agro',
 'CROP': 'd:\\_py\\AgroIA_prod\\inputs\\data\\crop',
 'SOIL': 'd:\\_py\\AgroIA_prod\\inputs\\data\\soil\\ec3.soil',
 'WEATHER_RAW': 'D:\\_py\\Clima_AgroIA\\data-raw\\xavier-data',
 'RESULTS': 'd:\\_py\\AgroIA_prod\\output\\soja_pr\\Sensitivity Analysis',
 'OPTIMIZATION': 'd:\\_py\\AgroIA_prod\\output\\soja_pr\\Optimization'}

In [3]:
loader = BRDWGDLoader(data_dir=paths['WEATHER_RAW'])
print("BR-DWGD carregado de:", paths['WEATHER_RAW'])


BR-DWGD carregado de: D:\_py\Clima_AgroIA\data-raw\xavier-data


In [4]:
def montar_point_netcdf(cod_municipio, nome, lat, lon, elevation_m, df_yield_municipio, out_path):
    df_raw = loader.extract_point_series(lat, lon, start_date=DATA_INICIO, end_date=DATA_FIM)
    df_wofost = BRDWGDLoader.to_wofost_frame(df_raw)

    df_wofost['ano_safra'] = df_wofost['date'].apply(safra_ano_colheita)

    yield_by_year = df_yield_municipio.set_index('ano')['yield (kg/ha)'].to_dict()
    df_wofost['yield'] = df_wofost['ano_safra'].map(yield_by_year)

    ds = xr.Dataset(
        {
            'IRRAD': ('date', df_wofost['IRRAD'].values),
            'TMIN': ('date', df_wofost['TMIN'].values),
            'TMAX': ('date', df_wofost['TMAX'].values),
            'T2M': ('date', df_wofost['T2M'].values),
            'VAP': ('date', df_wofost['VAP'].values),
            'RAIN': ('date', df_wofost['RAIN'].values),
            'WIND': ('date', df_wofost['WIND'].values),
            'ano_safra': ('date', df_wofost['ano_safra'].values),
            'yield': ('date', df_wofost['yield'].values),
        },
        coords={'date': df_wofost['date'].values},
        attrs={
            'point_id': cod_municipio,
            'cod_municipio': cod_municipio,
            'municipio': nome,
            'latitude': float(lat),
            'longitude': float(lon),
            'elevation_m': float(elevation_m) if pd.notna(elevation_m) else np.nan,
            'country': 'Brazil',
            'state': 'Parana',
            'crop': 'soybean',
            'source': 'BR-DWGD',
        }
    )

    ds.to_netcdf(out_path)
    ds.close()


In [5]:
df_yield

,lat,lon,country,state,cod_municipio,county,elevation_m,ano,yield (kg/ha)
0,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2016,2760
1,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2017,3480
2,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2018,2640
3,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2019,3480
4,-23.301821,-50.338724,Brazil,Parana,4100103,Abatiá,609.72,2020,3480
...,...,...,...,...,...,...,...,...,...
3440,-23.748147,-53.569081,Brazil,Parana,4128807,Xambrê,314.40,2020,2000
3441,-23.748147,-53.569081,Brazil,Parana,4128807,Xambrê,314.40,2021,2400
3442,-23.748147,-53.569081,Brazil,Parana,4128807,Xambrê,314.40,2022,400
3443,-23.748147,-53.569081,Brazil,Parana,4128807,Xambrê,314.40,2023,3000


In [ ]:
n_ok, n_falha = 0, 0

for _, row in df_locs.iterrows():
    cod_municipio = row['cod_municipio']
    df_yield_municipio = df_yield[df_yield['cod_municipio'] == cod_municipio]

    if df_yield_municipio.empty:
        print(f"[pulado] {row['name']} ({cod_municipio}): sem dados de produtividade de soja no periodo.")
        continue

    out_path = os.path.join(paths['COMPLETO'], f"point_id_{cod_municipio}.nc")

    try:
        montar_point_netcdf(
            cod_municipio=cod_municipio,
            nome=row['name'],
            lat=row['lat'],
            lon=row['lon'],
            elevation_m=row.get('elevation_m', np.nan),
            df_yield_municipio=df_yield_municipio,
            out_path=out_path,
        )
        n_ok += 1
        print(f"[ok] {row['name']} ({cod_municipio}) -> {out_path}")
    except Exception as e:
        n_falha += 1
        print(f"[falha] {row['name']} ({cod_municipio}): {e}")

print(f"\nConcluido: {n_ok} municipios ok, {n_falha} falharam.")
loader.close()
